# Notebook 5: Natural Language → SPARQL

Here's where the AI comes in. We give Bedrock the ontology schema and a natural language question, and it generates an executable SPARQL query.


---
### Setup


In [ ]:
%graph_notebook_config --store-to NOTEBOOK_CONFIG --silent

In [ ]:
from workshop import *
import json
configure(neptune_config=json.loads(NOTEBOOK_CONFIG))

---
## Module 7: Natural Language → SPARQL

Here's where the AI comes in. We give Bedrock the ontology schema and a natural language question, and it generates a SPARQL query.

This is the **text-to-SPARQL** pattern - the LLM translates natural language into executable graph queries.


In [ ]:
# Verify Bedrock connectivity
# text_to_sparql() is provided by workshop.py using Claude Haiku 4.5
import boto3
sts = boto3.client("sts")
print(f"Account: {sts.get_caller_identity()['Account']}")
print(f"Region: {NEPTUNE_REGION}")

### Try It: "Show me all claims"

We send a natural language question to Bedrock along with the ontology schema. The LLM generates a SPARQL query, which we then execute against Neptune. The ontology constrains the LLM's output - it knows exactly what classes, properties, and relationships exist:


In [ ]:
# Ask Bedrock to generate SPARQL from natural language
question = "Show me all claims with their amounts and policies"
print(f'Question: "{question}"\n')

generated_sparql = text_to_sparql(question)
print(f"Generated SPARQL:\n{generated_sparql}\n")

# Execute the generated query against Neptune
rows = sparql_query(generated_sparql)
print(f"Results ({len(rows)} rows):")
for r in rows:
    print(f"  {r}")

### Try It: "Which claims were denied?"

A more complex question that requires the LLM to understand claim event semantics - it needs to match `eventType "denied"` and use `FILTER NOT EXISTS` to ensure "denied" is the latest event. The ontology's vocabulary guides the LLM to generate the correct pattern:


In [ ]:
# A more complex question requiring event semantics
question = "Which claims were denied? Show the claim ID, amount, and when they were denied."
print(f'Question: "{question}"\n')

generated_sparql = text_to_sparql(question)
print(f"Generated SPARQL:\n{generated_sparql}\n")

# Execute against Neptune
rows = sparql_query(generated_sparql)
print(f"Denied claims ({len(rows)}):")
for r in rows:
    amt = f"${float(r.get('claimAmount', r.get('amount', 0))):,.2f}"
    print(f"  {r.get('claimId', 'N/A'):>5}  {amt:>10}  {r.get('policyId', 'N/A')}")

→ Proceed to **06-Denied-Claims**
